In [ ]:
import os
device = "cuda"
os.environ["DIGNN_ENV"] = device

import sys
sys.path.append('..')

from DIGNN.data.utils import ase_db2AtomsData_list, update_basic_batch, update_cplt_graph
from DIGNN.utils.utils import AtomIndexMapper

from sklearn.model_selection import train_test_split
from torch_geometric.loader import DataLoader

import time
import numpy as np
from ase.db import connect

In [ ]:
def generate_Graphs_from_db(file_path:str) -> list[AtomsData]:
    """
    basic_graphs: list of AtomsData, 只包含元素、坐标、力、能量和拓扑结构
    """
    ase_db = connect(file_path)
    basic_graphs = []
    start_time = time.time()
    
    for i,mol in enumerate(ase_db.select()):
        mol = mol.toatoms()
        atom_numbers = mol.get_atomic_numbers()
        pos = mol.positions
        force = mol.get_forces()
        ene = np.array([mol.get_potential_energy()])
        if ene[0] > 20: continue
        if np.abs(force).max() > 200: continue
        
        data = AtomsData(atom = torch.from_numpy(atom_numbers).long(),
                         pos=torch.from_numpy(pos).float())
        data.force = torch.from_numpy(force).float()
        data.ene = torch.from_numpy(ene).float()
        
        basic_graphs.append(data)

        if i % 1000 == 0:
            print(f'sample:{i},time_cost:{time.time() - start_time}')
            start_time = time.time()
    
    return basic_graphs

basic_data = generate_Graphs_from_db('atoms.db')

In [ ]:
mapper = AtomIndexMapper(known_atom_nums=[5,67], device=device)

def atom_index_reset(basic_batch):
    for basic in basic_batch:
        basic.atom = mapper(basic.atom)
    return basic_batch

basic_data = atom_index_reset(basic_data)

In [ ]:
train_data, test_data = train_test_split(basic_data, test_size=0.2, random_state=42)

train = DataLoader(train_data, batch_size=16, follow_batch=['atom'])
test = DataLoader(test_data, batch_size=16, follow_batch=['atom'])

In [ ]:
pml_rcut = 2.5
pml_mnn = 16
iml_rcut = 5.0
iml_mnn = 32

basic_batch = update_basic_batch(train,pml_rcut=pml_rcut, pml_mnn=pml_mnn, 
                                iml_rcut=iml_rcut, iml_mnn=iml_mnn, store_device=device)
test_basic_batch = update_basic_batch(test, pml_rcut=pml_rcut, pml_mnn=pml_mnn, 
                                        iml_rcut=iml_rcut, iml_mnn=iml_mnn, store_device=device)


def get_cplt(basic):
    cplt = []
    for i,b in enumerate(basic):
        c = update_cplt_graph(b,
                            store_device=device, 
                            pos_grad=True, 
                            if_strip=True)
        cplt.append(c)
        
        if i % 100 == 0: print(f'cplt batch{i}')
    return cplt

In [ ]:
from DIGNN.nn import models as dgm
from DIGNN.nn.utils import init_weights

def create_model():
    feature_dim = {'atom': 128, 'bond': 128, 'angle': 64, 'dihedral': 32}
    model = dgm.dignn.DIGNN(encoder=dgm.Encoder(num_species=100,
                                                atom_dim=feature_dim['atom'],
                                                bond_dim=feature_dim['bond'],
                                                ang_dim=feature_dim['angle'],
                                                dih_dim=feature_dim['dihedral'],
                                                pml_rcut=pml_rcut+0.2,
                                                bondI_dim=feature_dim['bond'],
                                                iml_rcut=iml_rcut+0.2),
                    processor=gdm.GCN_Processor(atom_dim=feature_dim['atom'],
                                                bond_dim=feature_dim['bond'],
                                                ang_dim=feature_dim['angle'],
                                                dih_dim=feature_dim['dihedral'],
                                                pml=1,
                                                iml=4,
                                                residual=True,
                                                dropout=0.0,
                                                bondI_dim=feature_dim['bond']
                                                ), 
                    decoder=gdm.Decoder(dim=[feature_dim['atom'],64,1], 
                                        reduce_method='sum', 
                                        dropout=0.0),
                    ).to(device)
    model.apply(init_weights)
    return model
    

In [ ]:
loss_list = []

In [ ]:
val = [[],[]]
def validate(model, validate_basic_batch):
    batch_num = len(validate_basic_batch)
    model.eval()

    mae_ene_epoch = 0
    mae_force_epoch = 0

    mae_criterion = torch.nn.L1Loss()
    
    for basic in update_cplt_data.batch_iterator(validate_basic_batch):
        cplt = update_cplt_data.update_complete_graph(basic.clone(), device=device, pos_grad=True)
        cplt.strip_topo()
        
        energy = model(cplt)
        force = -torch.autograd.grad(outputs=energy, 
                                    inputs=cplt.pos, 
                                    grad_outputs=torch.ones_like(energy),
                                    create_graph=False,
                                    retain_graph=True,
                                    )[0]
        mae_ene = mae_criterion(energy.view(-1,1), cplt.ene.view(-1,1))
        mae_force = mae_criterion(force.view(-1,3), cplt.force.view(-1,3))


        val[0].append(mae_ene.item())
        val[1].append(mae_force.item())

        mae_ene_epoch += mae_ene.item()
        mae_force_epoch += mae_force.item()

    print(f'validate mae ene: {mae_ene_epoch/batch_num}, force: {mae_force_epoch/batch_num}')

In [ ]:
def train(model, batch_graphs, max_epoch=1000):
    runned_epoch = 500
    max_epoch -= runned_epoch

    batch_num = len(batch_graphs)

    criterion = torch.nn.MSELoss()
    optimizer = torch.optim.AdamW(model.parameters(), lr=1e-3, weight_decay=1e-2)
    scheduler = torch.optim.lr_scheduler.OneCycleLR(optimizer,
                                                    max_lr=1e-3,
                                                    total_steps=max_epoch*batch_num,
                                                    final_div_factor=1e+5,
                                                    )
    # scheduler = torch.optim.lr_scheduler.CosineAnnealingWarmRestarts(optimizer,
    #                                                                  T_0=int(max_epoch*batch_num*0.05),
    #                                                                  T_mult=2,
    #                                                                  eta_min=1e-8,
    #                                                                  last_epoch=-1)

    check_point = torch.load('checkpoint_500.pth')
    model.load_state_dict(check_point['model'])
    optimizer.load_state_dict(check_point['optimizer'])
    scheduler.load_state_dict(check_point['scheduler'])


    for epoch in range(max_epoch):
        epoch += runned_epoch
        model.train()
        loss_epoch = 0
        for i, batch in enumerate(batch_graphs):
            cplt = update_cplt_graph(batch.clone(),
                                        device=device, 
                                        pos_grad=True, 
                                        if_strip=True)
            energy = model(cplt)
            force = -torch.autograd.grad(outputs=energy, 
                                    inputs=cplt.pos, 
                                    grad_outputs=torch.ones_like(energy),
                                    create_graph=True,
                                    retain_graph=True,
                                    )[0]

            loss_ene = criterion(energy.view(-1,1), cplt.ene.view(-1,1))
            loss_force = criterion(force.view(-1,3), cplt.force.view(-1,3))
            loss = 0.1 * loss_ene + 1.0 * loss_force
        
            loss.backward(retain_graph=False)
            optimizer.step()

            optimizer.zero_grad()
            scheduler.step()
                        
            loss_list.append(loss.item())
            loss_epoch += loss.item()

        
        if epoch % 1 == 0:
            print(f'epoch: {epoch+1}, loss: {loss_epoch/batch_num}')
        if (epoch+1) % 10 == 0:
            validate(model, test_basic_batch)
        if (epoch+1) % 100 == 0:
            torch.save({'model': model.state_dict(),
                        'optimizer': optimizer.state_dict(),
                        'scheduler': scheduler.state_dict(),
                        'epoch': epoch,
                        }, f'checkpoint_{epoch+1}.pth')
    return model 

model = train(model, basic_batch,  max_epoch=1000)

In [ ]:
import gc

print(f"Current allocated: {torch.cuda.memory_allocated() / 1024 ** 2:.4f} MB")
print(f"Current reserver: {torch.cuda.memory_reserved() / 1024 ** 2:.4f} MB")

gc.collect()
torch.cuda.empty_cache()  # 释放显存缓存  
torch.cuda.reset_peak_memory_stats()

print(f"Current allocated: {torch.cuda.memory_allocated() / 1024 ** 2:.4f} MB")
print(f"Current reserver: {torch.cuda.memory_reserved() / 1024 ** 2:.4f} MB")

In [ ]:
from sklearn import metrics
from scipy.stats import gaussian_kde
from matplotlib import pyplot as plt
from scipy.stats import pearsonr

def error_caculation(target, pred, weight=None):
    """
    测量各种误差值
    Parameter
    -------
    target : 目标值.
    pred : 拟合预测值.

    Returns
    -------
    mae, mse, r2
    """
    
    mae=metrics.mean_absolute_error(target, pred, sample_weight=weight)
    # mse = metrics.mean_squared_error(target, pred)
    pcc = pearsonr(target, pred).correlation.tolist()[0]
    r2=metrics.r2_score(target,pred)
    
    #Pea = pearsonr(target_test.flatten(), pred_test.flatten())
    
    return mae, pcc, r2

# def density(x,y,xmin=None,xmax=None):
#     if xmin is not None:
#         mask = x>=xmin
#         x,y = x[mask],y[mask]
#     if xmax is not None:
#         mask = x<=xmax
#         x,y = x[mask],y[mask]
    
#     xy = np.vstack([x,y])
#     z = gaussian_kde(xy)(xy)
#     return x, y, z
def density(x, y, xmin=None, xmax=None, bins=100):
    if xmin is not None:
        mask = x >= xmin
        x, y = x[mask], y[mask]
    if xmax is not None:
        mask = x <= xmax
        x, y = x[mask], y[mask]
    # 计算二维直方图
    heatmap, xedges, yedges = np.histogram2d(x, y, bins=bins)
    # 转换为点坐标
    xidx = np.clip(np.digitize(x, xedges) - 1, 0, heatmap.shape[0]-1)
    yidx = np.clip(np.digitize(y, yedges) - 1, 0, heatmap.shape[1]-1)
    z = heatmap[xidx, yidx]
    return x, y, z
       
    
def plot_comparison(target, pred, x_min=0, x_max=3,colorbar_range=None, atom_num=None):
    x,y,z = density(target, pred, x_min, x_max)
    
    if colorbar_range is not None:
        min_mask = z < colorbar_range[0]
        max_mask = z > colorbar_range[1]
        
        z[min_mask] = colorbar_range[0]
        z[max_mask] = colorbar_range[1]
    
    mae, pcc, r2 = error_caculation(target, pred)
    if atom_num is not None:
        mae_peratom = metrics.mean_absolute_error(target, pred, sample_weight=1/atom_num) / np.mean(atom_num)
        text = f'Total: PCC={pcc:.3f}\nMAE={mae:.3f}\nR2={r2:.3f}\n\nPer atom: MAE={mae_peratom * 1e+3:.3f}meV\n'
    else:
        text = f'Total: PCC={pcc:.3f}\nMAE={mae*1000:.3f}\nR2={r2:.3f}\n'
    plt.text(0.95, 0.05, text, transform=plt.gca().transAxes,ha='right', va='bottom')
    
    #plt.plot(target, pred, 'o', markersize=1.5)#,label=[r'0'])
    plt.scatter(x,y,edgecolors='none',c=z,s=6,marker='o')#,cmap='viridis')
    plt.colorbar()
    #plt.legend()
    plt.tight_layout()
    # plt.show(block=False)
    pass


def plot_mono_comparison(target,pred,atom_num=None):
    mae, mse, r2 = error_caculation(target, pred)
    plt.scatter(target,pred,edgecolors='none',s=6,marker='o',label=f'Au{atom_num}')
    return mae, mse, r2

In [ ]:
def get_pred_target(batch_graphs):
    model.eval()
    
    pred_ene = []
    pred_force = []
    target_ene = []
    target_force = []
    
    for batch in update_cplt_data.batch_iterator(batch_graphs):
        cplt = update_cplt_graph(batch.clone(), 
                                device=device, 
                                pos_grad=True,
                                if_strip=True)
        energy = model(cplt)
        force = -torch.autograd.grad(energy, 
                                    cplt.pos, 
                                    grad_outputs=torch.ones_like(energy),
                                    retain_graph=True,
                                    create_graph=False)[0]

        pred_ene.append(energy.detach().cpu().view(-1,1).numpy())
        pred_force.append(force.detach().cpu().view(-1,3).numpy())
        
        target_ene.append(cplt.ene.detach().cpu().view(-1,1).numpy())
        target_force.append(batch.force.detach().cpu().view(-1,3).numpy())
    
    pred_ene = np.concatenate(pred_ene,axis=0)
    target_ene = np.concatenate(target_ene,axis=0)
    
    
    pred_force = np.concatenate(pred_force,axis=0)
    target_force = np.concatenate(target_force,axis=0)
    
    pred = [pred_ene.reshape(-1,1), pred_force.reshape(-1,3)]
    target = [target_ene.reshape(-1,1), target_force.reshape(-1,3)]
    return pred, target

test_i = 1
test_lst = [basic_batch, test_basic_batch]
pred, target = get_pred_target(test_lst[test_i])


test_atom_num = [train_data, test_data]
atoms_num = []
for d in test_atom_num[test_i]:
    atoms_num.append(d.atom.shape[0])

atoms_num = np.array(atoms_num)

In [ ]:
opt = 0
atoms_num = atoms_num

if opt == 0 : xrange = (-300,50)
if opt == 1 : xrange = (-100,100)
if opt == 2 : xrange = (-15,-1)

plt.figure(dpi=300,figsize=(7,5.3))


if opt == 0:
    plot_comparison(target[opt],pred[opt],atom_num=atoms_num, *xrange)
if opt == 1:
    plot_comparison(target[opt],pred[opt],atom_num=None, *xrange)
if opt == 2:
    plot_comparison((target[0]/atoms_num).reshape(-1,1),(pred[0]/atoms_num).reshape(-1,1), 
                    atom_num=None, *xrange)
    
plt.plot([*xrange], [*xrange], color='r', linestyle='-',linewidth=0.5)  # 添加 y=x 的线

plt.xlim(*xrange)
plt.ylim(*xrange)
plt.xticks()
plt.yticks()
#plt.grid(False)
plt.xlabel('target')
plt.ylabel('pred')